# Ke Jin — alert policy v1 and automated checks

Frozen model: LightGBM no_dc, run aki_ml_v1_28e0b1fb83, Platt calibration. This notebook completes artifact/leakage checks and a prespecified alert-policy experiment without retraining.

## System-level protocol decision

The supplied scaffold selected on train OOF and only confirmed on validation. The user's project requirement selects warning thresholds on validation and incorporates data-cutoff stability. This version therefore uses deterministic **patient-level validation selection (70%) and confirmation (30%) roles**. Train OOF is a diagnostic comparison only, computed after the choice is sealed. The original Demo I policy is reproduced and preserved.

The full validation baseline was already viewed in Demo I and during restoration. The confirmation role is an internal policy check, **not an untouched final test or clinical validation**. No test features/labels are loaded.

## Modules and safeguards

- Checks A1–A12: src/aki_eval/checks.py.
- Fixed-origin cutoff replay: src/aki_eval/replay.py, reusing the frozen feature engine.
- Stateful alert events, separate from risk probabilities: src/aki_eval/policies.py.
- Config: configs/alert_policy_v1.yaml; finite rules and thresholds fixed before evaluation.
- Immutable local prespec and choice records: src/aki_eval/protocol.py.
- Orchestration: src/aki_eval/study.py; this notebook is a thin, rerunnable entry point.

Artifacts are aggregate-only, under a versioned artifacts/handoff/kejin/ run. No Git commit, push or PR is performed automatically. A local prespec hash is not a claim of public preregistration or pre-results Git history.

## v2 measurement amendment (before candidate choice/confirmation)

The initial v1 run stopped on an extra exact-risk replay control, before a choice was sealed or any confirmation. Stored features reproduced saved risks, but the exported event representation yielded small feature differences and some tree-branch risk differences. The policy grid and quality constraints were not changed.

Version v2 reports an explicit zero-lag representation control and compares lagged inputs against both stored-snapshot and zero-lag replay references. These are end-to-end sensitivity checks; numerical export/aggregation effects cannot all be causally attributed to withheld clinical measurements. The original sampled A10 tolerance remains unchanged. Feature-replay review remains a deployment blocker, and the candidate remains unlocked.


In [ ]:
from pathlib import Path
import sys
from threadpoolctl import threadpool_limits

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "alert_policy_v1.yaml").is_file())
sys.path.insert(0, str(REPO_ROOT / "src"))
from aki_eval.study import KeJinStudy

THREAD_LIMIT = threadpool_limits(limits=2)
study = KeJinStudy(REPO_ROOT)
print("Run:", study.out.name)
print("Prespec SHA256:", study.data.prespec["prespec_sha256"])


## Part 0 — Reproduce frozen Demo I

Both train-OOF and full-validation alert metrics must match the saved policy before the new experiment proceeds. This baseline reproduction does not tune any candidate.


In [ ]:
display(study.reproduce_baseline())


## Part A — Artifact and temporal checks

A1 unique keys; A2 probabilities; A3 schema/version/model/config hashes; A4 prediction coverage; A5 patient splits; A6 selected stay mapping; A7 chronology; A8 label windows/eligibility; A9 pre-onset origins; A10 at least 2,000 feature replays plus future injection; A11 urine-specific future truncation; A12 fitting declarations and actual OOF fold roles.

Failure stops policy selection. Details contain counts only. A10 independently replays dynamic features but receives admission-static/service inputs from recorded snapshots; raw source timing for those inputs is unavailable. A12 checks historical manifest assertions and saved fold roles; it does not independently re-audit inaccessible training data.


In [ ]:
display(study.checks())


## Part B.1 — Immutable prespecification

The config, data/model hashes and evaluation-code hashes are written before a candidate grid is evaluated. Existing records cannot be replaced with different inputs, settings or code. The finite grid compares sustain, cooldown, episode suppression, and substantial-risk-rise repeat rules; unvalidated confidence and extra multi-horizon thresholds are deliberately excluded.

Hourly k-score sustain means observed high duration of at least k-1 h. Gaps reset sustain and episodes. Early missing input is unscorable, not zero risk.

Selection minimises false alerts per scorable monitored patient-day under detection, lead, PPV, alert-burden and cutoff-stability constraints. Non-target-event patient burden cannot increase. Ties favour fewer affected non-target-event patients, then simpler rules and higher thresholds.


In [ ]:
display(study.prespecification())


## Part B.2 — Validation selection and training-OOF diagnostics

Every choice is made using validation-selection patients. A zero-lag replay control and 1/2 h input cutoffs keep prediction origin and label window fixed. Rolling windows stay anchored at origin; laboratory reporting delay still applies. Hourly service uses its last recorded cutoff-visible state; raw service-event timing is not supplied.

Cutoff comparisons report paired coverage, probability changes, threshold flips, alert-event flips and patient alert-presence flips. Both point-warning and patient-alert stability affect selection. The risk model and calibrated probabilities are never trained or altered.

After choice.json is sealed, the requested policy_grid_train_oof.csv and policy_candidates_train_oof.csv are generated as **diagnostics, not selection**. Validation-selection grids/representatives and trade-off figures are also saved. No feasible candidate is reported honestly rather than relaxing criteria.


In [ ]:
display(study.select())


## Part B.3 — Fixed confirmation and paired patient bootstrap

Compare only the sealed candidate and frozen Demo I on the other validation patients. Use 1,000 **paired patient-level** bootstrap resamples, including confidence intervals for metric differences. Confirmation checks the same prespecified constraints and cutoff sensitivity. Failure does not trigger retuning.

A completed confirmation is reused on a rerun with the same immutable choice. Different settings require a new labelled development cycle and appropriate untouched evaluation data; rerunning code does not create a new independent validation sample.

The candidate JSON always has threshold.locked=false and requires a stateful alert adapter. The current live loader rejects it, preventing a sustained/cooldown/episode policy from silently degrading into the website's instantaneous threshold comparison. Original policy.json, fitted weights and src/aki_ml remain unchanged.


In [ ]:
display(study.confirm())


## Summary and hand-back

The summary reports actual trade-offs and the confirmation outcome. Definitions match original Demo I: exposure is scorable monitored hours / 24, unknown-outcome alerts count toward burden but not PPV, and non_aki denotes patients without a target event in the evaluated monitoring/horizon risk set.

Source notebook outputs are cleared for sharing. Executed aggregate-only copies and versioned CSV/JSON/PNG files stay local under artifacts/handoff/kejin/. The root LATEST.json identifies the run. Required files include checks.csv, prespec.json, choice.json, both OOF diagnostic tables, validation-selection tables, policy_confirmation_validation.csv, policy_v1_candidate.json, cutoff tables, trade-off figures and SUMMARY.md.

No final-test evaluation or live clinical-policy activation is claimed.


In [ ]:
print(study.summary())
from scripts.summarize_alert_policy import finalize_reports
print(finalize_reports(study.out))
from IPython.display import Image, display
display(Image(filename=str(study.out / "policy_tradeoff_validation_selection.png")))


## Recorded v2 result — kejin-validation-policy-v2-21d980db7c292219

- A1–A12 passed within their recorded schema/sample/provenance evidence scope.
- Validation roles: 3,420 selection patients and 1,530 confirmation patients.
- Selected candidate: threshold 23%, 12 h cooldown, unchanged risk model.
- Confirmation false alerts/scorable patient-day: 0.424 → 0.246 (about 42% lower).
- Target-event detection: 76.95% → 73.56%.
- Median first-correct-alert lead: 12.82 → 12.09 h.
- PPV: 33.78% → 33.19%; candidate below the prespecified 1/3 floor.
- Outcome: **not confirmed; no retuning and no website-policy replacement**.
- Zero-lag representation differences remain explicitly reported; live use needs feature-replay review and a stateful alert adapter.
- Final test and clinical evaluation were not performed; no Git pre-registration, commit, push or PR is claimed.


**Post-confirmation diagnosis:** [Replay numerical diagnosis and follow-up plan](../handoff/REPLAY-NUMERIC-DIAGNOSIS.md) localizes cumulative-arithmetic and static-input precision defects. Whole-validation and original-cutoff-sample controls preserve the original selection and PPV non-confirmation. Individual predictions can differ, and a separate rapid-rise cache defect requires repair. No code cells, fixed choice, criteria, confirmation record or active policy were changed by this note.
